In [ ]:
import joblib
from pathlib import Path
import pandas as pd
import numpy as np
import sklearn

In [ ]:
data = Path("../datasets/umu")
results_path = Path("artifacts/01-Changed_and_added_model-results")
random_seed = 42
n_splits = 5

In [ ]:
# Praparation
# We assume the dataset has been downloaded and unzipped manually

df = pd.read_excel(data / "umu" / "tcp_nokia_20240325.xlsx")

df = df[
    ["Column7","Column8","Column14","Column15","Column42","Column43","Column45",
     "Column46","Column47","Column48","Column87","Column88","Column78","Column79"]
]
df.columns = df.iloc[0]
df = df[1:]

# convert all columns to numeric
for column in df.columns:
    if df[column].dtype == "object":
        df[column] = pd.to_numeric(df[column], errors="coerce")

df.columns = [col.replace("nas_value_nr5g_", "") for col in df.columns]

df = df.dropna()  # drom the ~2 rows with NaN
df = df.loc[:, df.nunique() > 1]  # keep only columns with more than one uniqe value

In [ ]:
#Feature generation
import math

def lat_lon_to_meters(origin_lat, origin_lon, point_lat, point_lon) -> tuple[float, float]:
    """Works "fine" for distances less than 100km"""

    R = 6_378_137 # Earth's radius in meters

    origin_lat_rad = math.radians(origin_lat) # Convert latitude and longitude from degrees to radians
    delta_lat_rad = math.radians(point_lat - origin_lat)
    delta_lon_rad = math.radians(point_lon - origin_lon)

    delta_meters_lat = delta_lat_rad * R # Calculate distance in the latitude direction (North-South)

    delta_meters_lon = delta_lon_rad * R * math.cos(origin_lat_rad) # Calculate distance in the longitude direction (East-West)

    return delta_meters_lat, delta_meters_lon

origin_lat, origin_lon = df.gpsd_tpv_lat.min(), df.gpsd_tpv_lon.min()

df[["target_x", "target_y"]] = df.apply(
    lambda row: lat_lon_to_meters(origin_lat, origin_lon, row["gpsd_tpv_lat"], row["gpsd_tpv_lon"]),
    axis=1,
    result_type="expand",
)

df.drop(columns=["gpsd_tpv_lat", "gpsd_tpv_lon"], inplace=True)

targets = ["target_x", "target_y"] # Find target column(s)

features, targets = df.drop(targets, axis=1), df[targets] # X are features, y are target(s)

In [ ]:
#Split generation
from sklearn import model_selection
groups = None

cv = model_selection.KFold(
    n_splits=n_splits,
    shuffle=True,
    random_state=random_seed,
)

indices = []

for train_indices, test_indices in cv.split(features, targets, groups):
        indices.append((train_indices, test_indices))


In [ ]:
class PredefinedSplit(model_selection.BaseCrossValidator):
    """Simple cross-validator for predefined train-test splits."""

    def __init__(self, indices_pairs: list[tuple[np.ndarray, np.ndarray]]):
        self.idx_pairs = indices_pairs

    def get_n_splits(self, X=None, y=None, groups=None):
        """Return the number of splitting iterations in the cross-validator"""
        return len(self.idx_pairs)

    def split(self, X, y=None, groups=None):
        """Generate indices to split data into training and test set."""
        for train_idx, test_idx in self.idx_pairs:
            yield train_idx, test_idx

In [ ]:
#Train&Evaluate
from sklearn.ensemble import RandomForestRegressor 
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import make_scorer, root_mean_squared_error
cv = PredefinedSplit(indices)

estimators = [
    RandomForestRegressor(random_state=42),
    KNeighborsRegressor()
]
params=[
    {
        "n_estimators": [10, 50, 100, 250, 400], 
        "max_depth": [5, 10, 30, 50, 150, 200, None]
    },
    {
        "n_neighbors": [3, 5, 10], 
        "weights": ["uniform", "distance"], 
        "p": [1, 2], 
        "leaf_size": [10, 15, 30], 
        "metric": ["minkowski", "euclidean"] 
    }
]

for index in range(2):
    gs = model_selection.GridSearchCV(
        estimator = estimators[index],
        param_grid = params[index],
        n_jobs = 4,
        error_score = "raise",
        refit = True,
        scoring = make_scorer(root_mean_squared_error, greater_is_better=False),
        cv = cv,
    )
    
    gs.fit(features, targets)
    
    results_df = pd.DataFrame(gs.cv_results_)
    
    # Select key columns to display
    cols_to_show = [
        'params',
        'mean_test_score',
        'std_test_score',
        'rank_test_score',
        'mean_fit_time',
        'mean_score_time',
    ]
    
    # Print as a table
    print(results_df[cols_to_show].to_string(index=False))
    Path(results_path).mkdir(parents=True, exist_ok=True)
    joblib.dump(gs.best_estimator_, results_path / f"Model_{estimators[index].__class__.__name__}-KFoldSplit.pkl") # Note it's only possible to get the best estimator, as the framework uses a modified version of the class to save all the models
    joblib.dump(results_df, results_path / f"Results_{estimators[index].__class__.__name__}-KFoldSplit.pkl")